# Global E-Commerce Giants 2015–2026 — Starter Notebook

This notebook is the fastest way to get familiar with the dataset. It covers:

1. **Load & inspect** the seven CSV tables.
2. **EDA**: regional distribution, who grew fastest, who got crushed the hardest.
3. **Visualize** the equal-weighted Global E-Commerce Index.
4. **Peer clustering** using only the derived `company_metrics.csv` features.
5. **Baseline ML**: predict next-month return of the sector index from past returns.

Everything runs end-to-end against the CSVs in `../data/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.3f}".format)
DATA = "../data"

companies   = pd.read_csv(f"{DATA}/companies.csv")
prices      = pd.read_csv(f"{DATA}/prices_daily.csv", parse_dates=["date"])
financials  = pd.read_csv(f"{DATA}/financials_annual.csv")
macro       = pd.read_csv(f"{DATA}/macro_indicators.csv")
metrics     = pd.read_csv(f"{DATA}/company_metrics.csv")
index_df    = pd.read_csv(f"{DATA}/ecommerce_index.csv", parse_dates=["date"])
dictionary  = pd.read_csv(f"{DATA}/data_dictionary.csv")

for name, df in [
    ("companies", companies), ("prices_daily", prices), ("financials_annual", financials),
    ("macro_indicators", macro), ("company_metrics", metrics), ("ecommerce_index", index_df),
]:
    print(f"{name:<22s} shape={df.shape}")

## 1· Who's in the universe?

In [ ]:
by_region = companies.groupby(["region", "country_code"]).size().rename("n").reset_index()
by_region = by_region.sort_values(["region", "n"], ascending=[True, False])
print(by_region.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
companies["region"].value_counts().sort_values().plot.barh(ax=ax)
ax.set_title("Companies per region")
ax.set_xlabel("# companies")
plt.tight_layout()

In [ ]:
# Glance at the master table
companies[["company_id", "name", "ticker", "country_code", "segment", "founded_date"]].head(10)

## 2· Who grew fastest? Who fell furthest?

In [ ]:
winners = metrics.sort_values("cagr", ascending=False)[
    ["company_id", "country_code", "trading_days", "cagr", "annualized_volatility", "max_drawdown"]
].head(10)
print("Top 10 by price CAGR:")
print(winners.to_string(index=False))

print("\nLargest drawdowns:")
print(metrics.sort_values("max_drawdown")[
    ["company_id", "country_code", "max_drawdown", "cagr", "annualized_volatility"]
].head(10).to_string(index=False))

In [ ]:
# Scatter: volatility vs CAGR — the classic risk-return picture
fig, ax = plt.subplots(figsize=(9, 6))
for region, sub in metrics.dropna(subset=["cagr"]).groupby("region"):
    ax.scatter(sub["annualized_volatility"], sub["cagr"], label=region, s=60, alpha=0.75)
for _, r in metrics.dropna(subset=["cagr"]).iterrows():
    ax.annotate(r["company_id"], (r["annualized_volatility"], r["cagr"]),
                fontsize=7, alpha=0.7)
ax.axhline(0, color="grey", lw=0.5)
ax.set_xlabel("Annualized volatility")
ax.set_ylabel("CAGR (price)")
ax.set_title("Risk ↔ return across e-commerce giants, 2015–2026")
ax.legend(fontsize=8)
plt.tight_layout()

## 3· The Global E-Commerce Index

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(index_df["date"], index_df["index_level"], color="#1f77b4", lw=1.6, label="Index level (rebased 100)")
ax1.set_ylabel("Index level")
ax2 = ax1.twinx()
ax2.plot(index_df["date"], index_df["constituents"], color="#bbb", lw=0.8, label="# constituents")
ax2.set_ylabel("# constituents")
ax1.set_title("Equal-weighted Global E-Commerce Index")
ax1.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# Annual return table
annual = (index_df.assign(year=index_df["date"].dt.year)
                  .groupby("year")
                  .agg(start=("index_level", "first"), end=("index_level", "last")))
annual["return"] = annual["end"] / annual["start"] - 1
annual["return_pct"] = annual["return"].map(lambda x: f"{x:+.1%}")
annual[["return_pct"]]

## 4· Peer clustering from derived features

We cluster the 43 companies using their derived metrics (`cagr`, `annualized_volatility`, `max_drawdown`, `return_90d`, `return_1y`, `revenue_cagr`, `net_margin`) and see whether segment / region drop out naturally.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

feat_cols = ["cagr", "annualized_volatility", "max_drawdown",
             "return_90d", "return_1y", "revenue_cagr", "net_margin"]
X = metrics[feat_cols].copy()
X = X.fillna(X.median())
Xs = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=4, n_init=20, random_state=0).fit(Xs)
metrics["cluster"] = kmeans.labels_

pcs = PCA(n_components=2, random_state=0).fit_transform(Xs)
metrics["pc1"], metrics["pc2"] = pcs[:, 0], pcs[:, 1]

fig, ax = plt.subplots(figsize=(10, 7))
for cl, sub in metrics.groupby("cluster"):
    ax.scatter(sub["pc1"], sub["pc2"], s=60, alpha=0.8, label=f"cluster {cl}")
for _, r in metrics.iterrows():
    ax.annotate(r["company_id"], (r["pc1"], r["pc2"]), fontsize=7, alpha=0.7)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("E-commerce peer clusters (PCA of derived metrics)")
ax.legend()
plt.tight_layout()

In [ ]:
# What do the clusters mean? Look at cluster centroids in original units.
centroids = (metrics.groupby("cluster")[feat_cols + ["company_id"]]
             .agg({c: "mean" for c in feat_cols} | {"company_id": "count"})
             .rename(columns={"company_id": "n"}))
centroids

## 5· Baseline ML: forecast next-month return of the sector index

A simple supervised regression. Features are *past* monthly returns of the index; target is the *next* month's return. We use a chronological train/test split (no leakage) and report RMSE plus naive baselines.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

monthly = (index_df.set_index("date")["index_level"]
                  .resample("ME").last()
                  .pct_change()
                  .dropna()
                  .rename("ret")
                  .to_frame())

# 6 lag features
for k in range(1, 7):
    monthly[f"lag_{k}"] = monthly["ret"].shift(k)
monthly["target"] = monthly["ret"].shift(-1)  # predict NEXT month
ds = monthly.dropna().copy()

feat = [f"lag_{k}" for k in range(1, 7)]
split = int(len(ds) * 0.8)
X_tr, X_te = ds[feat].iloc[:split], ds[feat].iloc[split:]
y_tr, y_te = ds["target"].iloc[:split], ds["target"].iloc[split:]

results = {}
for name, model in [("Ridge", Ridge(alpha=1.0)),
                    ("GBR",   GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0))]:
    model.fit(X_tr, y_tr)
    rmse = mean_squared_error(y_te, model.predict(X_te)) ** 0.5
    results[name] = rmse

# Naive baselines
results["naive_zero"] = mean_squared_error(y_te, np.zeros_like(y_te)) ** 0.5
results["naive_lag1"] = mean_squared_error(y_te, X_te["lag_1"]) ** 0.5
results["naive_train_mean"] = mean_squared_error(y_te, np.full_like(y_te, y_tr.mean())) ** 0.5

print("Test-set RMSE (lower is better):")
for k, v in sorted(results.items(), key=lambda kv: kv[1]):
    print(f"  {k:<18s} {v:.4f}")

## 6· Quick joins cookbook

A few common patterns you'll want.

In [ ]:
# Join prices to companies for a country filter
india_prices = prices.merge(
    companies[["company_id", "country_code"]], on="company_id"
).query("country_code == 'IN'")
print(india_prices["company_id"].nunique(), "Indian companies,",
      f"{len(india_prices):,} price rows")

# Per-company latest revenue alongside country macro context
latest_fy = (financials.dropna(subset=["total_revenue"])
             .sort_values(["company_id", "fiscal_year"])
             .groupby("company_id").tail(1))

macro_latest = (macro.query("indicator_name == 'internet_users_pct'")
                .sort_values(["country_code", "year"])
                .groupby("country_code").tail(1)
                .rename(columns={"value": "internet_users_pct"})
                [["country_code", "internet_users_pct"]])

panel = (companies[["company_id", "country_code", "region", "segment"]]
         .merge(latest_fy[["company_id", "fiscal_year", "total_revenue"]], on="company_id")
         .merge(macro_latest, on="country_code", how="left"))
panel.head(10)

## Where to go next

Some ideas to push toward a competition-grade submission:

- **Multi-horizon forecasting** with exogenous features from `macro_indicators.csv` and the company’s own derived metrics.
- **Survival / drawdown** classification: which features in the first 2 years of trading predict a 50%+ drawdown later?
- **Cross-region tilt**: build a long-Indian / short-US portfolio and benchmark against the equal-weighted index.
- **NLP enrichment**: use the `wiki_extract` field to embed companies and combine textual similarity with numerical clustering.

Pull requests and issues welcome — see the README for licensing & citation.